# LeetCode 1781 — Sum of Beauty of All Substrings bằng Python

Link đề: https://leetcode.com/problems/sum-of-beauty-of-all-substrings/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Counting

Skeleton LeetCode:

```python
class Solution:
    def beautySum(self, s: str) -> int:
        ...
```

## 1. Mục tiêu học

1. Hiểu "độ đẹp" (beauty) của một chuỗi: tần suất **lớn nhất** trừ tần suất **nhỏ nhất** (chỉ xét các ký tự **có mặt**).
2. Từ vét cạn `O(n³)` (đếm lại từng chuỗi con) chuyển sang **mở rộng dần** chuỗi con từ mỗi điểm đầu, cập nhật bảng đếm `O(1)`.
3. Duy trì **max** tăng dần và **min** bằng bảng "số ký tự có tần suất f" (frequency of frequencies) để đạt `O(n²)`.
4. Dùng **mảng tiền tố đếm** (prefix count) như một cách khác.
5. Đối chiếu các cách trên chuỗi ngẫu nhiên có seed cố định và đo thời gian ở `n = 500`.

## 2. Đề bài & ví dụ

**Độ đẹp** của một chuỗi là hiệu giữa tần suất của ký tự xuất hiện **nhiều nhất** và ký tự xuất hiện **ít nhất** (trong số các ký tự có mặt). Ví dụ độ đẹp của `"abaacc"` là `3 − 1 = 2` (`a` 3 lần, `b` 1 lần).

Cho chuỗi `s`, trả về **tổng độ đẹp của mọi chuỗi con** của `s`.

**Ví dụ 1:** `s = "aabcb"` → `5`. Các chuỗi con có độ đẹp khác 0: `"aab"`, `"aabc"`, `"aabcb"`, `"abcb"`, `"bcb"` — mỗi chuỗi có độ đẹp 1.

**Ví dụ 2:** `s = "aabcbaa"` → `17`.

**Ràng buộc:** `1 ≤ s.length ≤ 500`, `s` chỉ gồm chữ thường.

In [1]:
import random
import time
from collections import Counter
from typing import List

examples = [("aabcb", 5), ("aabcbaa", 17)]
for s, exp in examples:
    print(f"s={s!r} -> kỳ vọng {exp}")

s='aabcb' -> kỳ vọng 5
s='aabcbaa' -> kỳ vọng 17


## 3. Kiến thức nền

### 3.1 `Counter` và độ đẹp của một chuỗi

In [2]:
def beauty(t):
    c = Counter(t)
    return max(c.values()) - min(c.values())

for t in ["abaacc", "aab", "abc", "zzz"]:
    print(t, dict(Counter(t)), "-> độ đẹp", beauty(t))

abaacc {'a': 3, 'b': 1, 'c': 2} -> độ đẹp 2
aab {'a': 2, 'b': 1} -> độ đẹp 1
abc {'a': 1, 'b': 1, 'c': 1} -> độ đẹp 0
zzz {'z': 3} -> độ đẹp 0


### 3.2 Liệt kê chuỗi con theo điểm đầu `i` và điểm cuối `j`

Cố định `i`, cho `j` chạy `i, i+1, …, n−1`: chuỗi con `s[i..j]` **mở rộng thêm đúng một ký tự** mỗi bước ⇒ có thể cập nhật bảng đếm thay vì đếm lại.

### 3.3 Mảng đếm 26 phần tử

`cnt[ord(ch) − 97] += 1`. Lấy max/min trên các phần tử **dương**: `max(cnt)`, `min(v for v in cnt if v)`.

In [3]:
cnt = [0] * 26
for ch in "aab":
    cnt[ord(ch) - 97] += 1
print(max(cnt), min(v for v in cnt if v))

2 1


### 3.4 Tần suất của tần suất (`freq`)

`freq[f]` = số ký tự đang có tần suất đúng bằng `f`. Khi ký tự `c` tăng từ `f` lên `f + 1`: `freq[f] −= 1`, `freq[f+1] += 1`. Min tần suất là `f` nhỏ nhất có `freq[f] > 0` — chỉ thay đổi theo những cách rất hạn chế (xem mục 6).

## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1** `"aabcb"`, với điểm đầu `i = 0`:

| j | chuỗi con | đếm | max | min | độ đẹp |
|---|---|---|---|---|---|
| 0 | a | a:1 | 1 | 1 | 0 |
| 1 | aa | a:2 | 2 | 2 | 0 |
| 2 | aab | a:2, b:1 | 2 | 1 | 1 |
| 3 | aabc | a:2, b:1, c:1 | 2 | 1 | 1 |
| 4 | aabcb | a:2, b:2, c:1 | 2 | 1 | 1 |

Tổng cho `i = 0` là 3; `i = 1` góp `"abcb"` (1); `i = 2` góp `"bcb"` (1) ⇒ tổng 5.

Quan sát: khi `j` tăng, bảng đếm chỉ đổi **một** ô (+1) — không cần đếm lại cả chuỗi con.

In [4]:
s = "aabcb"
for i in range(len(s)):
    print(i, [(s[i:j + 1], beauty(s[i:j + 1])) for j in range(i, len(s))])

0 [('a', 0), ('aa', 0), ('aab', 1), ('aabc', 1), ('aabcb', 1)]
1 [('a', 0), ('ab', 0), ('abc', 0), ('abcb', 1)]
2 [('b', 0), ('bc', 0), ('bcb', 1)]
3 [('c', 0), ('cb', 0)]
4 [('b', 0)]


## 5. Cách 1 — Brute Force / baseline: đếm lại từng chuỗi con

In [5]:
def beauty_bruteforce(s):
    n = len(s)
    return sum(beauty(s[i:j + 1]) for i in range(n) for j in range(i, n))

for s, exp in examples:
    got = beauty_bruteforce(s); print(got, "kỳ vọng", exp); assert got == exp

5 kỳ vọng 5
17 kỳ vọng 17


**Độ phức tạp:** `n(n+1)/2` chuỗi con, mỗi chuỗi đếm `O(n)` ⇒ `O(n³)` ≈ `2·10⁷` với `n = 500` — chậm trong Python (vài giây).

**Nút thắt:** đếm lại từ đầu mỗi chuỗi con, trong khi các chuỗi con cùng điểm đầu lồng nhau.

In [6]:
s = "".join(random.Random(0).choice("abc") for _ in range(150))
t = time.perf_counter(); beauty_bruteforce(s); print(f"brute force n=150: {(time.perf_counter()-t)*1000:.0f} ms (n=500 chậm hơn ~37 lần)")

brute force n=150: 16 ms (n=500 chậm hơn ~37 lần)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (mở rộng dần):** cố định `i`, tăng `j`, cập nhật `cnt[s[j]] += 1`. Mỗi chuỗi con tốn `O(26)` để tính max/min ⇒ `O(26 n²)`.

**Insight 2 (max đơn điệu):** với `i` cố định, khi `j` tăng thì `max` **không giảm** và chỉ có thể tăng đúng 1 khi ký tự vừa thêm đạt tần suất mới: `mx = max(mx, cnt[c])`.

**Insight 3 (min bằng `freq`):** khi thêm ký tự `c` có tần suất cũ `f`:
- Nếu `f == 0` (ký tự mới xuất hiện) **thì** `mn = 1`.
- Nếu `f == mn` và sau khi chuyển `freq[f]` về `0` **thì** `mn = f + 1` (chính `c` giờ là một trong những ký tự ít nhất).
- Ngược lại `mn` không đổi.

⇒ Mỗi bước `O(1)`, tổng `O(n²)`.

**Invariant:** sau khi xử lý `s[i..j]`, `mx`/`mn` đúng bằng tần suất lớn nhất/nhỏ nhất (trên ký tự có mặt) của `s[i..j]`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mở rộng dần + `max`/`min` trên mảng 26

In [7]:
def beauty_extend(s):
    n = len(s); total = 0
    for i in range(n):
        cnt = [0] * 26
        for j in range(i, n):
            cnt[ord(s[j]) - 97] += 1
            total += max(cnt) - min(v for v in cnt if v)
    return total

print([beauty_extend(s) for s, _ in examples])

[5, 17]


**Code nộp LeetCode (Cách 2):** đổi tên thành `beautySum`. **Độ phức tạp:** `O(26 · n²)` ≈ `3,3·10⁶` thao tác cơ bản (nhưng `min` với generator có hằng số lớn).

### 7.2 Cách 3 — Mở rộng dần + `mx` tăng dần + `freq` cho `mn`, `O(n²)`

In [8]:
def beauty_freq(s):
    n = len(s); total = 0
    codes = [ord(ch) - 97 for ch in s]
    for i in range(n):
        cnt = [0] * 26
        freq = [0] * (n - i + 2)
        mx = mn = 0
        for j in range(i, n):
            c = codes[j]
            f = cnt[c]
            cnt[c] = f + 1
            if f:
                freq[f] -= 1
            freq[f + 1] += 1
            if f + 1 > mx:
                mx = f + 1
            if f == 0:
                mn = 1
            elif f == mn and freq[f] == 0:
                mn = f + 1
            total += mx - mn
    return total

print([beauty_freq(s) for s, _ in examples])

[5, 17]


**Code nộp LeetCode (Cách 3):** xem mục 9. **Độ phức tạp:** Time `O(n²)` (mỗi bước `O(1)`), Space `O(n)` cho `freq`.

### 7.3 Cách 4 — Mảng tiền tố đếm

`pre[k][c]` = số lần ký tự `c` xuất hiện trong `s[0..k−1]`. Tần suất của `c` trong `s[i..j]` là `pre[j+1][c] − pre[i][c]`. Mỗi chuỗi con tính `O(26)`. Không nhanh hơn Cách 2 về tiệm cận nhưng cho phép **truy vấn bất kỳ** `(i, j)` độc lập.

In [9]:
def beauty_prefix(s):
    n = len(s)
    pre = [[0] * 26]
    for ch in s:
        row = pre[-1][:]; row[ord(ch) - 97] += 1; pre.append(row)
    total = 0
    for i in range(n):
        base = pre[i]
        for j in range(i, n):
            row = pre[j + 1]
            vals = [row[c] - base[c] for c in range(26) if row[c] - base[c]]
            total += max(vals) - min(vals)
    return total

print([beauty_prefix(s) for s, _ in examples])

[5, 17]


**Độ phức tạp Cách 4:** Time `O(26 · n²)`, Space `O(26 · n)`.

### 7.4 Cách 5 — Mở rộng dần chỉ trên các ký tự **có mặt**

Biến thể thực dụng của Cách 2: giữ danh sách các ký tự đã xuất hiện (tối đa 26, thường ít hơn nhiều), max/min chỉ duyệt trên chúng.

In [10]:
def beauty_present(s):
    n = len(s); total = 0
    for i in range(n):
        cnt = {}
        for j in range(i, n):
            cnt[s[j]] = cnt.get(s[j], 0) + 1
            vals = cnt.values()
            total += max(vals) - min(vals)
    return total

print([beauty_present(s) for s, _ in examples])

[5, 17]


**Độ phức tạp Cách 5:** `O(k · n²)` với `k` = số ký tự khác nhau (`≤ 26`).

## 8. TRACE từng bước

Trace Cách 3 với điểm đầu `i = 0` trên **Ví dụ 2** `"aabcbaa"`:

In [11]:
def trace_freq_row(s, i):
    cnt = [0] * 26; freq = [0] * (len(s) + 2); mx = mn = 0; row_total = 0
    for j in range(i, len(s)):
        c = ord(s[j]) - 97; f = cnt[c]; cnt[c] = f + 1
        if f: freq[f] -= 1
        freq[f + 1] += 1
        mx = max(mx, f + 1)
        if f == 0: mn = 1
        elif f == mn and freq[f] == 0: mn = f + 1
        row_total += mx - mn
        present = {chr(97 + k): v for k, v in enumerate(cnt) if v}
        print(f"j={j} thêm {s[j]!r}: đếm={present}, freq={ {k: v for k, v in enumerate(freq) if v} }, mx={mx}, mn={mn}, độ đẹp={mx - mn}, cộng dồn={row_total}")
    return row_total

r0 = trace_freq_row("aabcbaa", 0)
assert r0 == sum(beauty("aabcbaa"[0:j + 1]) for j in range(7))

j=0 thêm 'a': đếm={'a': 1}, freq={1: 1}, mx=1, mn=1, độ đẹp=0, cộng dồn=0
j=1 thêm 'a': đếm={'a': 2}, freq={2: 1}, mx=2, mn=2, độ đẹp=0, cộng dồn=0
j=2 thêm 'b': đếm={'a': 2, 'b': 1}, freq={1: 1, 2: 1}, mx=2, mn=1, độ đẹp=1, cộng dồn=1
j=3 thêm 'c': đếm={'a': 2, 'b': 1, 'c': 1}, freq={1: 2, 2: 1}, mx=2, mn=1, độ đẹp=1, cộng dồn=2
j=4 thêm 'b': đếm={'a': 2, 'b': 2, 'c': 1}, freq={1: 1, 2: 2}, mx=2, mn=1, độ đẹp=1, cộng dồn=3
j=5 thêm 'a': đếm={'a': 3, 'b': 2, 'c': 1}, freq={1: 1, 2: 1, 3: 1}, mx=3, mn=1, độ đẹp=2, cộng dồn=5
j=6 thêm 'a': đếm={'a': 4, 'b': 2, 'c': 1}, freq={1: 1, 2: 1, 4: 1}, mx=4, mn=1, độ đẹp=3, cộng dồn=8


Lưu ý bước thêm `a` cuối (`j = 6`): `a` từ 3 lên 4 ⇒ `mx = 4`; `mn` vẫn 1 (`c` chỉ có 1). Tổng các hàng `i = 0..6` cho đáp án:

In [12]:
rows = [sum(beauty("aabcbaa"[i:j + 1]) for j in range(i, 7)) for i in range(7)]
print("đóng góp theo điểm đầu i:", rows, "-> tổng", sum(rows)); assert sum(rows) == 17

đóng góp theo điểm đầu i: [8, 4, 3, 1, 1, 0, 0] -> tổng 17


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 3** — `O(n²)` với `freq`.

In [13]:
class Solution:
    def beautySum(self, s: str) -> int:
        n = len(s)
        codes = [ord(ch) - 97 for ch in s]
        total = 0
        for i in range(n):
            cnt = [0] * 26
            freq = [0] * (n - i + 2)
            mx = mn = 0
            for j in range(i, n):
                c = codes[j]
                f = cnt[c]
                cnt[c] = f + 1
                if f:
                    freq[f] -= 1
                freq[f + 1] += 1
                if f + 1 > mx:
                    mx = f + 1
                if f == 0:
                    mn = 1
                elif f == mn and freq[f] == 0:
                    mn = f + 1
                total += mx - mn
        return total

for s, exp in examples:
    got = Solution().beautySum(s); print(got, exp); assert got == exp

5 5
17 17


## 10. Vì sao thuật toán đúng

Cố định `i`. Xét bước thêm ký tự `c` có tần suất cũ `f` vào `s[i..j−1]` để được `s[i..j]`. Chỉ tần suất của `c` thay đổi (`f → f+1`).

**Max:** tần suất lớn nhất mới `= max(mx_cũ, f + 1)` vì mọi tần suất khác giữ nguyên.

**Min (trên ký tự có mặt):**
- `f = 0`: `c` mới xuất hiện với tần suất 1, mà mọi tần suất `≥ 1` ⇒ `mn = 1`.
- `f > 0`, `f > mn_cũ`: ký tự đạt `mn_cũ` vẫn còn (không phải `c`) ⇒ `mn` không đổi.
- `f = mn_cũ`: nếu còn ký tự khác có tần suất `f` (`freq[f] > 0` sau cập nhật) ⇒ `mn` giữ `f`; nếu không, mọi ký tự có mặt đều có tần suất `> f`, và `c` giờ có `f + 1` — không có ai nằm giữa `f` và `f + 1` ⇒ `mn = f + 1`.

Ba trường hợp phủ hết ⇒ invariant giữ; cộng `mx − mn` cho mọi `(i, j)` ⇒ đúng tổng định nghĩa. ∎

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s) ≤ 500`, `Σ = 26`.

| Cách | Time | Space |
|---|---|---|
| 1. Đếm lại mỗi chuỗi con | `O(n³)` | `O(Σ)` |
| 2. Mở rộng + max/min 26 | `O(Σ n²)` | `O(Σ)` |
| 3. Mở rộng + freq | `O(n²)` | `O(n)` |
| 4. Tiền tố đếm | `O(Σ n²)` | `O(Σ n)` |
| 5. Chỉ ký tự có mặt | `O(k n²)` | `O(k)` |

In [14]:
rng = random.Random(1781)
big = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(500))
for name, f in [("Cách 2", beauty_extend), ("Cách 3", Solution().beautySum), ("Cách 4", beauty_prefix), ("Cách 5", beauty_present)]:
    t = time.perf_counter(); r = f(big); print(f"{name}: {r} ({(time.perf_counter()-t)*1000:.0f} ms)")

Cách 2: 1100490 (122 ms)


Cách 3: 1100490 (14 ms)


Cách 4: 1100490 (162 ms)


Cách 5: 1100490 (63 ms)


## 12. Bộ test

In [15]:
ALL = {"extend": beauty_extend, "freq": beauty_freq, "prefix": beauty_prefix, "present": beauty_present,
       "submit": Solution().beautySum}
def run_all(s):
    ref = beauty_bruteforce(s)
    for k, f in ALL.items():
        assert f(s) == ref, (k, s)
    return ref

for s, exp in examples:
    assert run_all(s) == exp
edge = [("a", 0), ("ab", 0), ("aab", 1), ("aaaa", 0), ("abab", 2), ("aabb", 2)]
for s, exp in edge:
    assert run_all(s) == exp, s
rng = random.Random(17810)
for _ in range(600):
    s = "".join(rng.choice(rng.choice(["ab", "abc", "abcdef", "abcdefghijklmnopqrstuvwxyz"])) for _ in range(rng.randint(1, 30)))
    run_all(s)
print("Ví dụ, edge case, 600 test ngẫu nhiên: OK")

Ví dụ, edge case, 600 test ngẫu nhiên: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tính `min` trên cả 26 ô (kể cả ký tự **không có mặt**)

In [16]:
def wrong_min_zero(s):
    n = len(s); total = 0
    for i in range(n):
        cnt = [0] * 26
        for j in range(i, n):
            cnt[ord(s[j]) - 97] += 1
            total += max(cnt) - min(cnt)                 # SAI: min luôn 0
    return total

s, exp = examples[0]
print("sai :", wrong_min_zero(s), "| đúng:", exp)

sai : 21 | đúng: 5


**Vì sao sai:** chỉ xét ký tự **có mặt**; ô 0 làm min luôn bằng 0 ⇒ cộng cả `max`. **Sửa:** `min(v for v in cnt if v)`.

### Lỗi 2 — Cập nhật `mn` sai khi tần suất của ký tự ít nhất tăng

In [17]:
def wrong_mn_update(s):
    n = len(s); total = 0
    for i in range(n):
        cnt = [0] * 26; mx = mn = 0
        for j in range(i, n):
            c = ord(s[j]) - 97; cnt[c] += 1
            mx = max(mx, cnt[c])
            mn = 1 if cnt[c] == 1 else mn               # SAI: không bao giờ tăng mn
            total += mx - mn
    return total

s = "abab"                                      # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", wrong_mn_update(s), "| đúng:", Solution().beautySum(s), "| brute:", beauty_bruteforce(s))

sai : 3 | đúng: 2 | brute: 2


**Vì sao sai:** ở `"abab"`, khi cả `a` và `b` đều lên 2, min phải là 2 (độ đẹp 0), code sai vẫn giữ 1. **Sửa:** dùng `freq` để biết khi nào mọi ký tự có tần suất `mn` đã "lên hạng".

### Lỗi 3 — Chỉ cộng các chuỗi con độ dài ≥ 2 hoặc bỏ qua chuỗi con lồng nhau

In [18]:
def wrong_distinct_substrings(s):
    seen = set(); total = 0
    for i in range(len(s)):
        for j in range(i, len(s)):
            t = s[i:j + 1]
            if t not in seen:                            # SAI: chuỗi con trùng nội dung ở vị trí khác vẫn phải tính
                seen.add(t); total += beauty(t)
    return total

s = "aabaab"                                   # ví dụ tự dựng (đối chiếu lại đề gốc): "aab" xuất hiện 2 lần
print("sai :", wrong_distinct_substrings(s), "| đúng:", Solution().beautySum(s), "| brute:", beauty_bruteforce(s))

sai : 13 | đúng: 14 | brute: 14


**Vì sao sai:** "mọi chuỗi con" tính theo **vị trí**: trong `"aabaab"`, chuỗi `"aab"` (độ đẹp 1) xuất hiện ở hai vị trí và phải được cộng hai lần. **Sửa:** không khử trùng.

### Lỗi 4 — Đặt `cnt` ngoài vòng `i` (không reset khi đổi điểm đầu)

In [19]:
def wrong_no_reset(s):
    n = len(s); total = 0; cnt = [0] * 26                # SAI: phải tạo mới cho mỗi i
    for i in range(n):
        for j in range(i, n):
            cnt[ord(s[j]) - 97] += 1
            total += max(cnt) - min(v for v in cnt if v)
    return total

s, exp = examples[0]
print("sai :", wrong_no_reset(s), "| đúng:", exp)

sai : 30 | đúng: 5


**Vì sao sai:** đếm tích lũy của các chuỗi con trước bị cộng dồn sang điểm đầu mới. **Sửa:** khởi tạo `cnt` (và `freq`, `mx`, `mn`) **bên trong** vòng `i`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tổng một đại lượng trên **mọi chuỗi con**", `n ≤ 500–5000` ⇒ cố định điểm đầu, **mở rộng dần** điểm cuối, cập nhật `O(1)`/`O(Σ)`.
- Cần min/max của tần suất khi chỉ tăng ⇒ max tự cập nhật; min dùng **freq-of-freq**.
- Bảng chữ nhỏ (26) ⇒ `O(26)` mỗi bước thường chấp nhận được.
- Luôn nhớ "chỉ ký tự có mặt" khi lấy min.

## 15. Biến thể của bài

1. **Tổng max tần suất** hoặc **tổng min tần suất** riêng lẻ trên mọi chuỗi con (Bài tập 4).
2. **Chuỗi con có độ đẹp đúng bằng k**: đếm thay vì cộng.
3. **2262 Total Appeal of A String**: tổng số ký tự khác nhau của mọi chuỗi con — làm được `O(n)` bằng đóng góp từng vị trí.
4. **1248 / 992**: đếm mảng con thỏa điều kiện tần suất (cửa sổ trượt).
5. **Giới hạn n = 10⁵**: cần ý tưởng khác hẳn (đóng góp theo ký tự) — vượt mức Medium.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Counter mỗi chuỗi con | `O(n³)` | `O(Σ)` | Kiểm chứng |
| 2. Mở rộng + 26 | Cập nhật một ô, max/min 26 | `O(Σn²)` | `O(Σ)` | Dễ viết |
| 3. Mở rộng + freq | max tăng, min qua freq | `O(n²)` | `O(n)` | **Bản nộp** |
| 4. Tiền tố đếm | Hiệu hai hàng tiền tố | `O(Σn²)` | `O(Σn)` | Truy vấn đoạn bất kỳ |
| 5. Ký tự có mặt | dict nhỏ | `O(kn²)` | `O(k)` | Bảng chữ thực tế nhỏ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `"abc"`; (b) `"aaab"`; (c) `"abcabc"`; (d) `"xyyzzz"`.

**Bài 2 (tính tay):** Liệt kê mọi chuỗi con có độ đẹp khác 0 của `"aaab"` và tổng.

**Bài 3 (điền TODO):** Hoàn thiện `row_beauty(s, i)` trả tổng độ đẹp của các chuỗi con bắt đầu tại `i` (Cách 2).

**Bài 4 (tự viết lại):** Viết `sum_max_freq(s)` trả tổng **tần suất lớn nhất** của mọi chuỗi con, `O(n²)`.

**Bài 5 (mini challenge):** Viết `count_beauty_k(s, k)` đếm số chuỗi con có độ đẹp **đúng bằng** `k`, `O(n²)`.

In [20]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def row_beauty_todo(s, i):
    cnt = [0] * 26; total = 0
    for j in range(i, len(s)):
        # TODO: cnt[...] += 1; total += max(cnt) - min(các ô dương)
        pass
    return total

In [21]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def sum_max_freq(s):
    # TODO: mở rộng dần, mx = max(mx, cnt[c])
    pass

def count_beauty_k(s, k):
    # TODO: như Cách 3, đếm khi mx - mn == k
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: liệt kê chuỗi con có ≥ 2 ký tự khác nhau. Mức 2: độ đẹp mỗi chuỗi. Mức 3: (a) `0`; (b) `3`; (c) `5`; (d) `10` (với chuỗi dài hơn, hãy liệt kê bằng code `beauty(s[i:j+1])` để tự kiểm).

**Bài 2.** Mức 1: chuỗi con chứa `b` và ít nhất 2 chữ `a`. Mức 2: `"aab"` (1), `"aaab"` (2). Mức 3: tổng `3`.

**Bài 3.** Mức 1: `cnt[ord(s[j]) - 97] += 1`. Mức 2: `max(cnt)`. Mức 3: `min(v for v in cnt if v)`.

**Bài 4.** Mức 1: cố định `i`. Mức 2: `mx = max(mx, cnt[c])`. Mức 3: cộng `mx` mỗi bước; đối chiếu vét cạn.

**Bài 5.** Mức 1: dùng Cách 3. Mức 2: `if mx − mn == k: count += 1`. Mức 3: tổng `count_beauty_k(s, k)` với mọi `k` phải bằng `n(n+1)/2`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2, 3
b1 = [Solution().beautySum(t) for t in ["abc", "aaab", "abcabc", "xyyzzz"]]
print("Bài 1:", b1); assert b1 == [beauty_bruteforce(t) for t in ["abc", "aaab", "abcabc", "xyyzzz"]] and b1 == [0, 3, 5, 10]
nz = [(("aaab")[i:j + 1], beauty("aaab"[i:j + 1])) for i in range(4) for j in range(i, 4) if beauty("aaab"[i:j + 1])]
print("Bài 2:", nz, "tổng", sum(b for _, b in nz)); assert sum(b for _, b in nz) == 3
def row_beauty_todo(s, i):
    cnt = [0] * 26; total = 0
    for j in range(i, len(s)):
        cnt[ord(s[j]) - 97] += 1
        total += max(cnt) - min(v for v in cnt if v)
    return total
assert [row_beauty_todo("aabcbaa", i) for i in range(7)] == rows
print("Bài 3: OK", [row_beauty_todo("aabcb", i) for i in range(5)])

Bài 1: [0, 3, 5, 10]
Bài 2: [('aaab', 2), ('aab', 1)] tổng 3
Bài 3: OK [3, 1, 1, 0, 0]


In [23]:
# Bài 4, 5
def sum_max_freq(s):
    total = 0
    for i in range(len(s)):
        cnt = [0] * 26; mx = 0
        for j in range(i, len(s)):
            c = ord(s[j]) - 97; cnt[c] += 1
            if cnt[c] > mx: mx = cnt[c]
            total += mx
    return total

def count_beauty_k(s, k):
    n = len(s); res = 0
    for i in range(n):
        cnt = [0] * 26; freq = [0] * (n + 2); mx = mn = 0
        for j in range(i, n):
            c = ord(s[j]) - 97; f = cnt[c]; cnt[c] = f + 1
            if f: freq[f] -= 1
            freq[f + 1] += 1
            mx = max(mx, f + 1)
            if f == 0: mn = 1
            elif f == mn and freq[f] == 0: mn = f + 1
            res += (mx - mn == k)
    return res

rng = random.Random(4)
for _ in range(300):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 15)))
    subs = [s[i:j + 1] for i in range(len(s)) for j in range(i, len(s))]
    assert sum_max_freq(s) == sum(max(Counter(t).values()) for t in subs)
    k = rng.randint(0, 3)
    assert count_beauty_k(s, k) == sum(beauty(t) == k for t in subs)
    assert sum(count_beauty_k(s, kk) for kk in range(len(s) + 1)) == len(subs)
print("Bài 4, 5: OK |", sum_max_freq("aabcb"), [count_beauty_k("aabcbaa", k) for k in range(4)])

Bài 4, 5: OK | 21 [15, 10, 2, 1]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "mở rộng dần + freq-of-freq":**

```python
for i in range(n):
    cnt = [0]*26; freq = [0]*(n-i+2); mx = mn = 0
    for j in range(i, n):
        c = code[j]; f = cnt[c]; cnt[c] = f + 1
        if f: freq[f] -= 1
        freq[f+1] += 1
        mx = max(mx, f+1)
        if f == 0: mn = 1
        elif f == mn and freq[f] == 0: mn = f + 1
        total += mx - mn
```

**Checklist:**
- [ ] Min chỉ xét ký tự **có mặt** (không lấy 0)?
- [ ] Reset `cnt`, `freq`, `mx`, `mn` cho **mỗi** điểm đầu?
- [ ] Tính theo **vị trí** (không khử trùng chuỗi con)?
- [ ] `mn` tăng đúng lúc khi mọi ký tự ít nhất đã lên hạng?
- [ ] Đã test `n = 1`, toàn một chữ, xen kẽ `abab`, `n = 500`?